In [1]:
!nvidia-smi

Fri Jul 17 01:36:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install ninja

In [3]:
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load_inline

In [4]:
cuda_source = """
#include <cuda_runtime.h>
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>
#include <cuda.h>
#include <stdio.h>

__global__ void vec_sgemv_kernel(const float* __restrict__ mat, 
                                const float* __restrict__ vec, 
                                float* __restrict__ res, 
                                int M, int N) {
    extern __shared__ float smem[];

    int bid = blockIdx.x;
    if (bid >= M) return;

    int tid = threadIdx.x;
    int n_float4 = N /4;

    float4* mat_row = (float4*)&mat[bid * N];
    float4* vec_float4 = (float4*)vec;

    float partial_sum = 0.f;

    #pragma unroll 4
    for (int col = tid; col < n_float4; col += blockDim.x) {
        float4 mat_val = mat_row[col];
        float4 vec_val = vec_float4[col];
        partial_sum += (mat_val.x * vec_val.x) + (mat_val.y * vec_val.y) + 
                        (mat_val.z * vec_val.z) + (mat_val.w * vec_val.w);
    }

    int lane = tid % warpSize;
    int warp_id = tid / warpSize;

    for (int i = warpSize / 2; i > 0; i /= 2) {
        partial_sum += __shfl_down_sync(0xffffffff, partial_sum, i);
    }

    if (lane == 0) {
        smem[warp_id] = partial_sum;
    }
    __syncthreads();

    int num_warps = (blockDim.x + warpSize - 1) / warpSize;
    if (tid < num_warps) {
        partial_sum = smem[tid];
    } else {
        partial_sum = 0.f;
    }

    if (warp_id == 0) {
        for (int i = 1; i > 0; i >>= 1) {
            partial_sum += __shfl_down_sync(0xffffffff, partial_sum, i);
        }
        if (tid == 0) {
            smem[0] = partial_sum;
        }
    }
    __syncthreads();

    if (tid == 0) {
        res[bid] = smem[0];
    }
}

torch::Tensor gemv_forward(torch::Tensor mat, torch::Tensor vec) {
    auto mat_c = mat.contiguous();
    auto vec_c = vec.contiguous();

    const int M = mat_c.size(0);
    const int N = mat_c.size(1);

    auto res = torch::empty({M}, mat_c.options());

    int NUM_THREADS = 64;
    int warp_size = 32;

    dim3 block_size(NUM_THREADS);
    dim3 grid_size(M);
    size_t shared_mem_size = ((block_size.x + warp_size - 1) / warp_size) * sizeof(float);

    vec_sgemv_kernel<<<grid_size, block_size, shared_mem_size>>>(
        mat_c.data_ptr<float>(), vec_c.data_ptr<float>(), res.data_ptr<float>(), M, N);

    return res;
}
"""

In [5]:
cpp_source = "torch::Tensor gemv_forward(torch::Tensor mat, torch::Tensor vec);"

In [6]:
import os
build_dir = "/tmp/gelu_build"
os.makedirs(build_dir, exist_ok=True)

In [7]:
module = load_inline(
    name="gemv_vecotrized_wb",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["gemv_forward"],
    verbose=True,
    build_directory=build_dir
)

In [8]:
M, N = 4096, 8192
mat = torch.randn(M, N, device="cuda", dtype=torch.float32)
vec = torch.randn(N, device="cuda", dtype=torch.float32)

res = module.gemv_forward(mat, vec)
ref = torch.mv(mat, vec)

print("shape:", tuple(res.shape))
print("max abs diff:", (res - ref).abs().max().item())
print("allclose:", torch.allclose(res, ref, atol=1e-3, rtol=1e-4))

shape: (4096,)
max abs diff: 0.0001220703125
allclose: True
